### SMILES的粗粒度化

In [ ]:
# Import necessary libraries
import os
import sys
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display, SVG
import matplotlib.pyplot as plt

#smiles = "CC1=CC(=CC(=C1)C(=O)N(C(C)(C)C)NC(=O)C2=C(C3=C(C=C2)OCCC3)C)C  " 
#smiles = "C1CN(CCC1CCCC2CCN(CC2)CCO)CCO" 
#smiles = "CSCC[C@@H](C(=O)O)NC(=O)OCC1C2=CC=CC=C2C3=CC=CC=C13" 
#smiles = "CC1=CC(=C(C=C1SC2=CC(=C(C=C2C)O)C(C)(C)C)C(C)(C)C)O" 
#smiles = "C1[C@H]([C@H](OC2=CC(=CC(=C21)O)O)C3=CC(=C(C=C3)O)O)O" 
#smiles = "C1=CC=C(C=C1)OCC2=CC=CC3=C2C(=CC=C3)COC4=CC=CC=C4"
#smiles = "C1C2=CC=CC=C2C3=CC4=CC=CC=C4C=C31"
#smiles = "COC1=CC=CC=C1N2CCN(CC2)CC(COC3=CC=CC4=CC=CC=C43)O"
#smiles = "CCCCCCCCCC1CCCCC1"
#smiles = "CC(=O)OC1=C(C=C(C=C1)CC=C)OC"
#smiles = "CCCCCCC1=C2C=CC=CC2=CC3=CC=CC=C31"
#smiles = "C1=C2C(=CC3=C1C(=O)NC3=O)C(=O)NC2=O"
#smiles = "C1=CC=C2C=C3C(=O)C=CC(=O)C3=CC2=C1"
#smiles = "CC1=CC(=O)CC(C1)(C)C"
#smiles = "CC1=CNC(=O)NC1=O"
#smiles = "CC1=C(N=CN1)CO"
#smiles = "CCOC1=CC=C(C=C1)C(C)(C)COCC2=CC(=CC=C2)OC3=CC=CC=C3"
#smiles = "CC(C)(C)OC(=O)N1CCC(CC1)N1CCN(CC1)c1nsc(n1)-c1cccs1"
smiles = "C[C@@H](C1=CC2=C(C=C1)C=C(C=C2)OC)C(=O)O"    #19020
name = "PHPA_ester"

# Display the original molecule
mol = Chem.MolFromSmiles(smiles)
img = Draw.MolToImage(mol, size=(400, 300))
plt.figure(figsize=(5, 4))
plt.imshow(img)
plt.title(name)
plt.axis('off')
plt.show()

In [ ]:
!python model/coarse_graph.py --smiles "C1=CC(=CC=C1C[C@@H](C(=O)O)NC(=O)CN)O" --name 4335 --output_dir ./demo

### 数据集生成

In [ ]:
#数据集的生成
!python model/build_dataset.py --data_dir /mnt/e/code/spectra-scraper/NMR_spectral_data/organized_data \
    --shift_csv /mnt/e/code/spectra-scraper/NMR_spectral_data/shift_rdkit_data \
    --output_dir /mnt/e/code/NMRer-result/data3-1 --num_workers 16

In [ ]:
#数据集文件pt的读取
import torch
from model.coarse_graph import visualize_coarse_grained_graph 

graph_path = '/mnt/e/code/NMRer-result/data3-R/38.pt' 
graph_data = torch.load(graph_path,weights_only=True)
print(graph_data)

In [ ]:
!python model/s2n_predict.py \
    --checkpoint checkpoints_s2n/best_s2n_model.pt \
    --spectrum_csv /mnt/e/NN/GA/HSQ/standardized_nmr.csv \
    --elements "C=243,H=200,O=34,N=1,S=1,X=0" \
    --n_max 1000

       # --spectrum_csv /mnt/e/code/spectra-scraper/NMR_spectral_data/51.csv \

### 数据分析及其数据增强

In [ ]:
#结果输出
!python model/dataset_utils.py analyze \
    --data_dir /mnt/e/code/NMRer-result/data3-R \
    --output_dir analysis_report

In [ ]:
# 可视化节点分布和数量分布
from pathlib import Path
from model.dataset_utils import visualize_analysis_results

visualize_analysis_results(
    Path("analysis_report/su_distribution.csv"),
    Path("analysis_report/node_count_distribution.csv"),
    Path("analysis_report"),
)

In [ ]:
!python model/dataset_utils.py real_stats \
    --data_dir /mnt/e/code/NMRer-result/data3-R \
    --output_dir analysis_report

In [ ]:
!python model/dataset_utils.py split_ppm --data_dir /mnt/e/code/NMRer-result/data3-1 \
    --ok_dir /mnt/e/code/NMRer-result/data3-R \
    --bad_dir /mnt/e/code/NMRer-result/data3-B \
    --margin 0.0 --intensity_threshold 1e-6

In [ ]:
!python model/dataset_utils.py batch_merge \
    --data_dir /mnt/e/code/NMRer-result/data3-R/ \
    --output_dir /mnt/e/code/NMRer-result/data3_R_merged \
    --target_node_counts 220 240 250 260 275 280 300 325 350 400 450 500 550 375 475 425 225 275 375 50 60 70 80 90 100 120 140 160 180 200 \
    --num_per_target 100 \
    #--upsample_sus Alkyl_Cq Aryl_Substituted_Aro_C Carboxylic_Acid Aromatic_Bridgehead_C O_Substituted_Aro_C Alkyl_CH Ether_O Hydroxyl_O Alcohol_Ether_C\
    --upsample_factor 3.0

    # 220 240 260 280 300 350 400 450 500 30 40 50 60 70 80 90 100 120 140 160 180 200 

In [ ]:
# 核磁强度统计
!python model/dataset_utils.py intensity_stats \
    --data_dir /mnt/e/code/NMRer-result/data3-R/ \
    --output_dir analysis_report \
    --threshold 2.0

In [ ]:
!python model/z_library.py filter \
  -i z_library/subgraph_library.pt \
  -o z_library/subgraph_library_filtered.pt 

### G2S子图-->节点级核磁

In [ ]:
!python model/g2s_model.py /mnt/e/code/NMRer-result/data3-R \ 
    --hid 384 \
    --latent_dim 16 \
    --k_hop 3 \
    --beta 0.01 \
    --epochs 10 \
    --batch_size 32 \
    --lr 1e-4
# python g2s_model.py ../data3-R --hid 384 --latent_dim 16 --k_hop 3 --beta 0.01 --epochs 150 --batch_size 24 --lr 1e-5

In [ ]:
!python model/g2s_predict.py --input_path /mnt/e/code/NMRer-result/data1/21797.pt --model_path checkpoints_g2s/g2s_best_model.pt

In [ ]:
!python model/g2s_predict.py \
    --model_path checkpoints_g2s/g2s_best_model.pt \
    --input_path /mnt/e/code/NMRer-result/data3-R/23823.pt \
    # 23823 10915 5013 4335


### S2N 谱图/元素-->节点分布

In [ ]:
# 模型训练
!python model/s2n_model.py \
    /mnt/e/code/NMRer-result/data3-R/ \
    /mnt/e/code/NMRer-result/data3_R_merged \
    --epochs 10 \
    --batch_size 24 \
    --lr 0.00005 

# python model/s2n_model.py ../data3-R/ ../data3_R_merged --epochs 400 --batch_size 24 --lr 0.00005 

In [ ]:
# 验证
!python model/s2n_predict.py \
    --model_path checkpoints_s2n/best_s2n_model.pt \
    --input_path /mnt/e/code/NMRer-result/data3_R_merged/400_09.pt \
    #--num_samples 5

In [ ]:
# 预测
!python model/s2n_predict.py \
    --model_path checkpoints_s2n/best_s2n_model.pt \
    --spectrum_csv standardized_nmr.csv  \
    --elements "C=230, H=200, O=30, N=1, S=1, X=0" \
    --output_name test

       # --spectrum_csv /mnt/e/code/spectra-scraper/NMR_spectral_data/21797.csv \--spectrum_csv /mnt/e/NN/GA/HSQ/standardized_nmr.csv

### SMILES数据库

In [ ]:
!python model/smiles.py --inputs /mnt/e/code/NMRer-result/chembl_35_chemreps.txt --outdir /mnt/e/code/NMRer-result/smiles/35

In [ ]:
%cd /mnt/e/code/NMRer-result/smiles/35
!bash -lc 'LC_ALL=C find . -maxdepth 1 -type f -name "*.txt" -printf "%P\n" | sort | split -d -l 50000 - /tmp/split_; i=0; for lst in /tmp/split_*; do dir=$(printf "shard_%03d" "$i"); mkdir -p "$dir"; xargs -a "$lst" -I{} mv -- "{}" "$dir"/; i=$((i+1)); done; rm -f /tmp/split_*'

In [ ]:
!python model/build_dataset.py --data_dir smiles/shard_001 --output_dir smiles_out/shard_dataset_001 --num_workers 16

### 3  -hop子图库及其环境编码z

In [ ]:
# 子图数据库
!python model/z_library.py --pt_dir /mnt/e/code/NMRer-result/data3-R \
    --g2s_ckpt checkpoints_g2s/g2s_best_model.pt \
    --out z_library_3/subgraph_library.pt

In [ ]:
!python model/z_library.py --pt_dir smiles_out/shard_dataset_000 \
    --g2s_ckpt checkpoints_g2s/g2s_best_model.pt \
    --base_lib z_library/subgraph_library.pt \
    --merge_in_place

In [ ]:
!python model/z_library.py --pt_dir smiles_out/shard_dataset_001 \
    --g2s_ckpt checkpoints_g2s/g2s_best_model.pt \
    --base_lib z_library/subgraph_library.pt \
    --merge_in_place --mols_per_chunk 1500 \
    --num_workers 12 --prefetch_factor 2 --skip_role_index --amp

In [ ]:
!python model/z_library.py filter \
  -i z_library/subgraph_library.pt \
  -o z_library/subgraph_library_filtered.pt 

In [ ]:
# 统计分析
from model.z_library import analyze_template_sensitivity
analyze_template_sensitivity(
    pt_dir='/mnt/e/code/NMRer-result/data3-R',
    g2s_ckpt='checkpoints_g2s/g2s_best_model.pt',
    lib_path='z_library/subgraph_library.pt',
    out_csv='z_library/template_sensitivity.csv',
    g_samples=64, z_samples=64, device='cuda'
)

### 推理全过程

#spectrum_csv = Path("/mnt/e/NN/GA/LY/standardized_nmr.csv")
#elements_str = "C=600,H=350,O=27,N=7,S=4,X=0"

  --spectrum_csv /home/dofengqi/Graphnmr/standardized_nmr.csv \
  --elements "C=460,H=400,O=60,N=2,S=2,X=0" \

  --spectrum_csv /mnt/e/NN/GA/LY/standardized_nmr.csv \
  --elements "C=600,H=350,O=27,N=7,S=4,X=0" \

  --spectrum_csv /mnt/e/NN/GA/XLT/standardized_nmr.csv \
  --elements "C=300,H=355,O=62,N=7,S=8,X=0" \

  --spectrum_csv /mnt/e/NN/GA/MAS/standardized_nmr.csv \
  --elements "C=560,H=328,O=20,N=10,S=4,X=0" \

  --spectrum_csv /mnt/e/NN/GA/NT/standardized_nmr.csv \
  --elements "C=560,H=436,O=46,N=10,S=4,X=0" \

In [ ]:
!python3 test_v1.py \
  --spectrum_csv /home/dofengqi/Graphnmr/standardized_nmr.csv \
  --elements "C=460,H=400,O=60,N=2,S=2,X=0" \
  --output_dir test_results/HSQ-1 \
  --eval_nmr \
  --enable_hop1_adjust \
  --enable_layer3_approx_hop2 \
  --carbonyl_cycles 3 \
  --skeleton_cycles 1 \
  --final_smooth_sigma_ppm 1.2 \
  --final_smooth_passes 3

### 绘图    

In [ ]:
from model.visualize_inverse_results import (
    plot_layer3_su_distribution,
    plot_layer3_local_subgraphs,
)

plot_layer3_su_distribution("inverse_result/layer3_su_distribution.csv",
                            "inverse_result")

plot_layer3_local_subgraphs("inverse_result/layer3_nodes.csv",
                            "inverse_result",
                            num_examples=3)

In [ ]:
# 导入必要的库  
import pandas as pd  
import matplotlib.pyplot as plt  
def plot_ir_data(csv_file):  
    #df = pd.read_csv(csv_file, sep=';', header=None)  
    df = pd.read_csv(csv_file, sep=';', header=None)  
    plt.figure(figsize=(10, 6))  
    plt.plot(df[0], df[1], linestyle='-', color='b') 
    plt.title('NMR Spectrum')  
    plt.xlabel('Wavenumber (cm-1)')  
    plt.ylabel('Intensity (%)')  
    plt.grid()  
    plt.tight_layout()  
    plt.show()  
#csv_file = 'inverse_reconstructed_spectrum.csv'  
#csv_file = '/mnt/e/code/spectra-scraper/NMR_spectral_data/21797.csv'
#csv_file = '/mnt/e/NN/GA/HSQ/standardized_nmr.csv'
csv_file = '/mnt/e/NN/GA/XLT/standardized_nmr.csv'
plot_ir_data(csv_file)

In [ ]:
import pandas as pd  
import matplotlib.pyplot as plt  

txt_file = '/mnt/e/NN/GA/NMR.txt'
df = pd.read_csv(txt_file, sep='\s+', header=None, engine='python')

plt.figure(figsize=(10, 6))
plt.plot(df[0], df[1], 'b-', linewidth=1.5)
plt.title('NMR Spectrum')
plt.xlabel('Wavenumber (cm⁻¹)')
plt.ylabel('Intensity (%)')
plt.grid(True)
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt  

#graph_path = '/mnt/e/code/NMRer-result/data1/38.pt' 
#graph_path = 'coarse_graphs/83.pt' 
graph_path = '/mnt/e/code/NMRer-result/data2_merged/500_01.pt'
graph_data = torch.load(graph_path,weights_only=True)
#print(graph_data)
y = graph_data['y_spectrum']
plt.figure(figsize=(10, 6))
plt.plot(y, 'b-', linewidth=1.5)
plt.title('NMR Spectrum')
plt.xlabel('Wavenumber (cm⁻¹)')
plt.ylabel('Intensity (%)')
plt.grid(True)
plt.show()